In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn import model_selection, preprocessing, linear_model, naive_bayes, metrics, svm

## Datenaufbereitung
1. Laden Sie das *spam.csv*-File in das DataFrame *data*
2. Ändern Sie die Spaltenbezeichner: v1 -> **target** und v2 -> **message**
3. Transfomieren Sie den Inhalt der Spalte *message* in Kleinbuchstaben
4. Entfernen Sie Stoppwörter
5. Führen Sie Stemming und Lemmatisierung durch

In [6]:
data = pd.read_csv('spam.csv', encoding='ISO-8859-1')
data = data.rename(columns={'v1': 'target', 'v2': 'message'})
data['message'] = data['message'].apply(lambda x: x.lower())

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')

stop = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

def preprocess(text):
    tokens = nltk.word_tokenize(text)
    # Entfernen von Stopwords
    tokens = [token for token in tokens if token not in stop]
    # Lemmatization
    tokens = lemmatize_tokens(tokens)
    # Stemming
    tokens = stem_tokens(tokens)
    return " ".join(tokens)

data['message'] = data['message'].apply(preprocess)

# Ergebnis: 

ValueError: Length mismatch: Expected axis has 5 elements, new values have 2 elements

Das Dataset *data* mithilfe von `train_test_split()` in Trainings- und Validierungsdaten aufteilen:

In [30]:
train_x, valid_x, train_y, valid_y = train_test_split(..., test_size=.3)
print("Check shape: ", train_x.shape, valid_x.shape)

Check shape:  (3900,) (1672,)


Die Antworten (*ham* o. *spam*) in 1 oder 0 umwandeln:

In [5]:
encoder = preprocessing.LabelEncoder()
train_y = encoder.fit_transform(train_y)
valid_y = encoder.fit_transform(valid_y)

print("Encoder hat mit 0 und 1 zwei Klassen erkannt: ", encoder.classes_)
print("Die kodierten Antworten der Trainingsdaten: " , train_y)

Encoder hat mit 0 und 1 zwei Klassen erkannt:  ['ham' 'spam']
Die kodierten Antworten der Trainingsdaten:  [0 0 0 ... 1 0 0]


Vektorisierung der Daten mit TF-IDF. Mit der Größe von `max_features` kann experimentiert werden. Diese legt den Umfang des Vokabulars fest.

In [6]:
vectorizer = TfidfVectorizer(analyzer='word', max_features=5000)
vectorizer.fit(data['message'])

print("Vektor mit 5000 Einträgen: ",  vectorizer.get_feature_names_out().shape)
print("Vokabular: ", vectorizer.get_feature_names_out())

Vektor mit 5000 Einträgen:  (5000,)
Vokabular:  ['00' '000' '008704050406' ... 'ûï' 'ûïharri' 'ûò']


In [25]:
train_x_tfidf = vectorizer.transform(train_x)
valid_x_tfidf = vectorizer.transform(valid_x)
print("Check expected shape for train data: ", train_x_tfidf.shape)
print("Check expected shape for validation data: ", valid_x_tfidf.shape)

Check expected shape for train data:  (3900, 5000)
Check expected shape for validation data:  (1672, 5000)


Classifier trainieren und validieren:

In [24]:
clf = naive_bayes.MultinomialNB(alpha=0.2)
# fit the training dataset on the classifier
clf.fit(train_x_tfidf, train_y)

# predict the labels on validation dataset
predictions = clf.predict(valid_x_tfidf)

# retrieve accuracy score
metrics.accuracy_score(predictions, valid_y)

0.9856459330143541